In [1]:
# !pip install nltk --quiet
# !pip install spacy --quiet
# !python -m spacy download en
# !pip install datasets --quiet
# !pip install scikit-learn --quiet

In [2]:
from google.colab import drive
drive.mount("/content/drive/")

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [3]:
import spacy
import re
import pandas as pd
import joblib
from datasets import load_from_disk
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

In [4]:
ds = load_from_disk("/content/drive/MyDrive/Projects Preparation/Mail Spam detection/train")
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner", "attribute_ruler"])

In [5]:
nlp.pipe_names

['tok2vec', 'tagger', 'lemmatizer']

In [6]:
df = ds.to_pandas()
X = df[:]

In [7]:
X.head()

,message_id,text,label,label_text,subject,message,date
0,33214,any software just for 15 $ - 99 $ understandin...,1,spam,any software just for 15 $ - 99 $,understanding oem software\nlead me not into t...,2005-06-18
1,11929,perspective on ferc regulatory action client c...,0,ham,perspective on ferc regulatory action client c...,"19 th , 2 : 00 pm edt\nperspective on ferc reg...",2001-06-19
2,19784,wanted to try ci 4 lis but thought it was way ...,1,spam,wanted to try ci 4 lis but thought it was way ...,viagra at $ 1 . 12 per dose\nready to boost yo...,2004-09-11
3,2209,"enron / hpl actuals for december 11 , 2000 tec...",0,ham,"enron / hpl actuals for december 11 , 2000",teco tap 30 . 000 / enron ; 120 . 000 / hpl ga...,2000-12-12
4,15880,looking for cheap high - quality software ? ro...,1,spam,looking for cheap high - quality software ? ro...,"water past also , burn , course . gave country...",2005-02-13


In [8]:
X.columns

Index(['message_id', 'text', 'label', 'label_text', 'subject', 'message',
       'date'],
      dtype='object')

In [9]:
X.isna().sum()

,0
message_id,0
text,0
label,0
label_text,0
subject,0
message,0
date,0


In [10]:
X.shape

(31716, 7)

In [11]:
X, _= train_test_split(X, train_size=0.4, stratify=X['label'], random_state=42)

In [12]:
X.shape

(12686, 7)

In [13]:
X_train, X_test, y_train, y_test = train_test_split(X['message'], X['label'], test_size=0.3, stratify=X['label'], random_state=42)

In [14]:
print(f"{X_train.shape}|{X_test.shape}")

(8880,)|(3806,)


##Preprocessing

In [15]:
def cleaning(doc):
  return [token.lemma_ for token in doc
          if token.is_alpha and
          len(token.text) > 2]

###Training Sample Preprocessing

In [16]:
X_train = X_train.to_frame()

In [17]:
X_train['message'] = X_train['message'].str.lower()

In [18]:
docs = []
docs = list(nlp.pipe(X_train['message']))

/usr/local/lib/python3.12/dist-packages/spacy/pipeline/lemmatizer.py:187: UserWarning: [W108] The rule-based lemmatizer did not find POS annotation for one or more tokens. Check that your pipeline includes components that assign token.pos, typically 'tagger'+'attribute_ruler' or 'morphologizer'.
  warnings.warn(Warnings.W108)


In [19]:
X_train['tokens'] = [cleaning(doc) for doc in docs]
X_train.head()

,message,tokens
9899,"hallo ! be good , be kind , be humane , and ch...","[hallo, good, kind, humane, and, charitable, l..."
9179,"louise ,\ni understand there is concern that t...","[louise, understand, there, concern, that, the..."
27241,listed in domino directory\nyour message\nsubj...,"[listed, domino, directory, your, message, sub..."
31365,my only concern is that john ( i am told ) spo...,"[only, concern, that, john, told, spoke, gas, ..."
9429,to :\nandres almazan\ndon chew\njohn freeman\n...,"[andres, almazan, don, chew, john, freeman, ge..."


In [20]:
X_train['clean_text'] = X_train['tokens'].apply(lambda x: " ".join(x))
X_train.head()

,message,tokens,clean_text
9899,"hallo ! be good , be kind , be humane , and ch...","[hallo, good, kind, humane, and, charitable, l...",hallo good kind humane and charitable love you...
9179,"louise ,\ni understand there is concern that t...","[louise, understand, there, concern, that, the...",louise understand there concern that the value...
27241,listed in domino directory\nyour message\nsubj...,"[listed, domino, directory, your, message, sub...",listed domino directory your message subject s...
31365,my only concern is that john ( i am told ) spo...,"[only, concern, that, john, told, spoke, gas, ...",only concern that john told spoke gas and powe...
9429,to :\nandres almazan\ndon chew\njohn freeman\n...,"[andres, almazan, don, chew, john, freeman, ge...",andres almazan don chew john freeman geoge gau...


In [21]:
cv = CountVectorizer(ngram_range=(1,3))
X_train_cv = cv.fit_transform(X_train['clean_text'].values)

In [22]:
X_train_arr = X_train_cv.toarray()

###Test Sample Preprocessing

In [23]:
X_test = X_test.to_frame()

In [24]:
X_test['message'] = X_test['message'].str.lower()

In [25]:
test_docs = []
test_docs = list(nlp.pipe(X_test['message']))

/usr/local/lib/python3.12/dist-packages/spacy/pipeline/lemmatizer.py:187: UserWarning: [W108] The rule-based lemmatizer did not find POS annotation for one or more tokens. Check that your pipeline includes components that assign token.pos, typically 'tagger'+'attribute_ruler' or 'morphologizer'.
  warnings.warn(Warnings.W108)


In [26]:
X_test['tokens'] = [cleaning(doc) for doc in test_docs]
X_test.head()

,message,tokens
28384,pls review the executive list file on n : \ ho...,"[pls, review, the, executive, list, file, home..."
26807,in 1891 a collect call alyssa milano in 1824 s...,"[collect, call, alyssa, milano, small, world, ..."
4050,"band ago , made suffix . close ask believe . g...","[band, ago, made, suffix, close, ask, believe,..."
26670,important instructions for sending in expense ...,"[important, instructions, for, sending, expens..."
6715,"note : if this eamil is not fit for you , plea...","[note, this, eamil, not, fit, for, you, please..."


In [27]:
X_test['clean_text'] = X_test['tokens'].apply(lambda x: " ".join(x))
X_test.head()

,message,tokens,clean_text
28384,pls review the executive list file on n : \ ho...,"[pls, review, the, executive, list, file, home...",pls review the executive list file homedept nn...
26807,in 1891 a collect call alyssa milano in 1824 s...,"[collect, call, alyssa, milano, small, world, ...",collect call alyssa milano small world songs
4050,"band ago , made suffix . close ask believe . g...","[band, ago, made, suffix, close, ask, believe,...",band ago made suffix close ask believe game go...
26670,important instructions for sending in expense ...,"[important, instructions, for, sending, expens...",important instructions for sending expense rep...
6715,"note : if this eamil is not fit for you , plea...","[note, this, eamil, not, fit, for, you, please...",note this eamil not fit for you please reply w...


In [28]:
X_test_cv = cv.transform(X_test['clean_text'].values)

In [ ]:
X_test_arr = X_test_cv.toarray()

In [ ]:
X_test_arr[:1]

##Model Creation and Traning

In [ ]:
model = MultinomialNB()

In [ ]:
history = model.fit(X_train_arr, y_train)

##Saving Model

In [ ]:
joblib.dump(model, "/content/drive/MyDrive/Projects Preparation/Mail Spam detection/spam_detection.pkl")

##Loading Model

In [ ]:
model = joblib.load("/content/drive/MyDrive/Projects Preparation/Mail Spam detection/spam_detection.pkl")

##Prediction and Evaluation

In [ ]:
y_pre = model.predict(X_test_arr)

In [ ]:
print(classification_report(y_test, y_pre))

In [ ]:
texts = [
    '''We noticed you haven’t been opening our emails, and we don’t want to clutter your inbox.



If you still want updates, no action needed, you’ll continue receiving our emails.



If this email isn’t opened in 3 days, we’ll automatically unsubscribe you from our list. No worries, you can re-subscribe anytime here.'''
]

In [ ]:
tokens = []
tokens = list(nlp.pipe(texts))

In [ ]:
clean_token = [cleaning(token) for token in tokens]
clean_token

In [ ]:
clean_text = [" ".join(tokens) for tokens in clean_token]
clean_text

In [ ]:
texts_count = cv.transform(clean_text)
predictions = model.predict(texts_count)

In [ ]:
labels = ["Spam" if pred == 1 else "Ham" for pred in predictions]
print(labels)

##Practice Spacy

In [ ]:
lang = spacy.blank('en')

In [ ]:
text = """On 12th March 2025, Rahul Sharma met with Ananya Gupta at 10:30 AM in Kolkata
to discuss a data science project worth ₹50000. Rahul shared his contact details, including
his email rahul.sharma98@gmail.com and phone number +91-9876543210, while Ananya provided
hers as ananya.gupta@company.org and +91 91234 56789. They also looped in their colleague
Dr. Amit Verma, who joined remotely from Delhi and can be reached at amit.verma@research.in.
During the meeting, they reviewed 3 datasets containing over 1,20,000 records and finalized a
deadline of 30/04/2025. Later, Rahul sent 2 follow-up emails and attached 5 documents totaling 15 MB.
The team agreed to meet again after 7 days to track progress and ensure at least 95% accuracy
in their NLP model built using spaCy."""

In [ ]:
tokens = lang(text)

In [ ]:
nums = [n.text for n in tokens if n.like_num]
nums

In [ ]:
mails = [m.text for m in tokens if m.like_email]
mails

In [ ]:
lang.pipe_names

In [ ]:
eng_lang = spacy.load("en_core_web_sm")

In [ ]:
doc = eng_lang(text)

In [ ]:
nums = [n.text for n in doc if n.like_num]
nums

In [ ]:
mails = [m.text for m in doc if m.like_email]
mails

In [ ]:
names = [n.text for n in doc.ents if n.label_ == 'PERSON']
names

In [ ]:
o_name = [o.text for o in doc.ents if o.label_ == 'Time']
o_name

In [ ]:
for token in doc:
    print(f'''{token.text:<15} => {token.lemma_:<8}|{token.pos_:<8}|{spacy.explain(token.pos_):<15}|{token.is_alpha:<8}|{token.like_num:<8}|{token.is_stop:<8}''')